# 04 — Fine-tune con corrupciones vs DeMemte (backbone congelado)

**Tesis**: el approach estándar para ganar robustez es _aumentar el dataset con corrupciones_
y fine-tunear todo el modelo. Eso entrega un trade-off conocido: sube la accuracy en corrupt,
baja en clean. DeMemte propone otra ruta — congelar el backbone y delegar la corrección a un
gate de memoria que decide _cuándo_ intervenir. Esa diferencia debería romper el trade-off.

**Condición A** — `ResNet18 fine-tune completo` con corrupciones agresivas en entrenamiento.
**Condición B** — `DeMemte E5 frozen` (carga checkpoint del notebook 02).

`RUN_TRAINING=True` entrena la condición A desde cero (1× GPU). B no se reentrena nunca: solo
se carga el checkpoint de `notebooks/02_e5_winner/out/e5_best.pt`.

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'dememte').exists():
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('repo root:', ROOT)

In [ ]:
import json, shutil
from dataclasses import asdict

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from dememte.config import BaselineConfig, E5Config, resolve_data_dir
from dememte.data import build_loaders, seed_everything
from dememte.models import ResNetBaseline, make_dememte_e5
from dememte.training import train_baseline_phased
from dememte.evaluation import (
    evaluate_baseline_suite, evaluate_dememte_suite,
    signal_curve_rows, signal_curve_rows_baseline,
)
from dememte.io import save_checkpoint, load_checkpoint, write_json, write_csv, ensure_dir

RUN_TRAINING = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT = ensure_dir(ROOT / 'notebooks' / '04_finetune_vs_frozen' / 'out')
FT_CKPT = OUT / 'resnet_ft_best.pt'
E5_CKPT = ROOT / 'notebooks' / '02_e5_winner' / 'out' / 'e5_best.pt'
LEGACY_E5 = ROOT / 'experiments/atracctor/out/artifacts/dememte_e5_critical/seed_42/e5_combined_dropout_ood_tau_150/e5_combined_dropout_ood_tau_150_best.pt'
seed_everything(42)

## Datos

In [ ]:
data_dir = resolve_data_dir(BaselineConfig())
tr_loader, va_loader, te_loader, meta = build_loaders(
    data_dir=data_dir,
    batch_size=16,
    num_workers=2,
    val_ratio=0.2,
    split_seed=42,
    protocol='historical_trainval_resplit',
)
print(meta)

## Condición A — ResNet18 fine-tune completo con corrupciones agresivas

In [ ]:
ft_cfg = BaselineConfig(freeze_backbone=False)
ft_cfg.data_dir = data_dir
ft_cfg.train_corrupt_prob = 0.85  # aug agresivo para empujar robustez
ft_cfg.backbone_lr = 1e-4
ft_cfg.epochs_warmup = 2
ft_cfg.epochs_corrupt = 8
ft_cfg.epochs_joint = 12
print(json.dumps(asdict(ft_cfg), indent=2))

ft_model = ResNetBaseline(num_classes=ft_cfg.num_classes, freeze_backbone=False).to(device)
if RUN_TRAINING:
    ft_model, ft_best = train_baseline_phased(ft_model, tr_loader, va_loader, ft_cfg, device)
    save_checkpoint(ft_model, FT_CKPT, extra={'best_val': ft_best, 'config': asdict(ft_cfg)})
elif FT_CKPT.exists():
    payload = load_checkpoint(ft_model, FT_CKPT, device=device, strict=True)
    print('loaded FT:', FT_CKPT, '| best_val:', payload.get('best_val'))
else:
    print('WARNING: no FT checkpoint — flip RUN_TRAINING=True to train.')

## Condición B — DeMemte E5 frozen (carga, no reentrena)

In [ ]:
e5_cfg = E5Config()
e5_cfg.data_dir = data_dir
e5_model = make_dememte_e5(e5_cfg, device=device)
src = E5_CKPT if E5_CKPT.exists() else LEGACY_E5
if not src.exists():
    raise FileNotFoundError(f'no E5 checkpoint found at {E5_CKPT} or {LEGACY_E5}')
payload = load_checkpoint(e5_model, src, device=device, strict=True)
print('loaded E5:', src, '| best_val:', payload.get('best_val'))

## Evaluación paralela

In [ ]:
ft_metrics = evaluate_baseline_suite(ft_model, te_loader, device=device)
ft_clean = ft_metrics.pop('clean_record'); ft_corr = ft_metrics.pop('corruption_records')
ft_summary = {k: v for k, v in ft_metrics.items() if isinstance(v, (int, float))}
ft_summary['condition'] = 'A_resnet_ft_corrupt'

e5_metrics = evaluate_dememte_suite(e5_model, te_loader, device=device)
e5_clean = e5_metrics.pop('clean_record'); e5_corr = e5_metrics.pop('corruption_records')
e5_summary = {k: v for k, v in e5_metrics.items() if isinstance(v, (int, float, bool))}
e5_summary['condition'] = 'B_dememte_e5_frozen'

comparison = pd.DataFrame([ft_summary, e5_summary])
ft_trainable = sum(p.numel() for p in ft_model.parameters() if p.requires_grad)
e5_trainable = sum(p.numel() for p in e5_model.parameters() if p.requires_grad)
ft_total = sum(p.numel() for p in ft_model.parameters())
e5_total = sum(p.numel() for p in e5_model.parameters())
comparison['trainable_params'] = [ft_trainable, e5_trainable]
comparison['total_params'] = [ft_total, e5_total]
comparison['gap_clean_minus_corrupt'] = comparison['clean_acc'] - comparison['corrupt_acc_avg']
comparison.to_csv(OUT / 'comparison_table.csv', index=False)
comparison

## Curvas por corrupción

In [ ]:
ft_curves = signal_curve_rows_baseline('A_resnet_ft_corrupt', ft_clean, ft_corr)
e5_curves = signal_curve_rows('B_dememte_e5_frozen', 'DeMemte E5', e5_clean, e5_corr)
all_curves = pd.DataFrame(ft_curves + e5_curves)
all_curves.to_csv(OUT / 'comparison_curves.csv', index=False)

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharey=True)
for ax, corr in zip(axes.flat, ['gaussian_noise', 'pixel_mask', 'cutout', 'blur']):
    for variant, sub in all_curves[all_curves['corruption'] == corr].groupby('variant'):
        ax.plot(sub['severity'], sub['acc'], marker='o', label=variant)
    ax.set_title(corr); ax.grid(alpha=0.3); ax.set_xlabel('Severity'); ax.set_ylabel('Accuracy'); ax.legend()
ensure_dir(OUT / 'plots')
fig.savefig(OUT / 'plots' / 'robustness_curves.png', dpi=120, bbox_inches='tight')
plt.tight_layout(); plt.show()

## Scatter clean vs corrupt — ¿se rompe el trade-off?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for _, row in comparison.iterrows():
    ax.scatter(row['clean_acc'], row['corrupt_acc_avg'], s=180, label=row['condition'])
    ax.annotate(row['condition'], (row['clean_acc'], row['corrupt_acc_avg']),
                xytext=(8, 8), textcoords='offset points', fontsize=10)
ax.plot([0, 1], [0, 1], linestyle=':', color='gray', alpha=0.5, label='clean = corrupt')
ax.set_xlabel('Clean accuracy'); ax.set_ylabel('Corrupt accuracy (avg)')
ax.set_title('Trade-off clean ↔ corrupt')
ax.set_xlim(0.5, 1.0); ax.set_ylim(0.3, 0.8)
ax.grid(alpha=0.3); ax.legend(loc='lower right')
fig.savefig(OUT / 'plots' / 'scatter_tradeoff.png', dpi=120, bbox_inches='tight')
plt.show()

## Narrativa final

Lee la columna `gap_clean_minus_corrupt` de la tabla. Si el approach del gate de memoria
está cumpliendo su tesis, esperarías:

- Condición A (FT con corrupciones): clean cae respecto al baseline limpio y corrupt sube — el
  trade-off clásico. Gap moderado.
- Condición B (DeMemte E5 frozen): clean se mantiene alta **y** corrupt mejora. Gap menor o
  similar pero con ambos valores más altos.

El scatter visualiza esto: si B está arriba-y-a-la-derecha de A, no estamos en la misma frontera
de Pareto — el gate genuinamente rompe el trade-off.